# 01 - Data Exploration

This notebook explores the RetailRocket ecommerce dataset to understand:
- Data structure and quality
- User behavior patterns
- Event distributions
- Time patterns and seasonality

## Business Context
We're analyzing behavioral data from an ecommerce platform to predict customer churn.
Understanding the data is the first step toward building effective retention campaigns.

In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from src.data_loader import DataLoader

# Settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Data

First, download and load the RetailRocket dataset.

In [ ]:
# Initialize data loader
loader = DataLoader(data_dir='../data')
print(loader)

In [ ]:
# Download dataset (requires Kaggle API credentials)
# loader.download_from_kaggle()

# Or if you manually downloaded, just load directly:
# events = loader.load_events()

In [ ]:
# Load events data
events = loader.load_events()
print(f"Loaded {len(events):,} events")
events.head()

In [ ]:
# Basic info
events.info()

## 2. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values:")
print(events.isnull().sum())
print(f"\nMissing percentage:")
print((events.isnull().sum() / len(events) * 100).round(2))

In [ ]:
# Data summary
summary = loader.get_data_summary()
for key, value in summary.items():
    print(f"{key}: {value}")

## 3. Event Distribution Analysis

In [ ]:
# Event type distribution
event_counts = events['event'].value_counts()
print("Event distribution:")
print(event_counts)
print(f"\nPercentages:")
print((event_counts / len(events) * 100).round(2))

In [ ]:
# Visualize event distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
event_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('Event Type Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(event_counts, labels=event_counts.index, autopct='%1.1f%%',
            colors=['#3498db', '#2ecc71', '#e74c3c'])
axes[1].set_title('Event Type Proportion')

plt.tight_layout()
plt.savefig('../figures/event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Time-Based Analysis

In [ ]:
# Add time features
events['date'] = events['timestamp'].dt.date
events['hour'] = events['timestamp'].dt.hour
events['day_of_week'] = events['timestamp'].dt.dayofweek
events['week'] = events['timestamp'].dt.isocalendar().week

In [ ]:
# Daily event volume
daily_events = events.groupby(['date', 'event']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
daily_events.plot(ax=ax)
ax.set_title('Daily Event Volume Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Events')
ax.legend(title='Event Type')
plt.tight_layout()
plt.savefig('../figures/daily_events.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Hourly patterns
hourly = events.groupby(['hour', 'event']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
hourly.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Event Volume by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Count')
ax.legend(title='Event Type')
plt.tight_layout()
plt.savefig('../figures/hourly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Day of week patterns
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow = events.groupby(['day_of_week', 'event']).size().unstack(fill_value=0)
dow.index = day_names

fig, ax = plt.subplots(figsize=(10, 4))
dow.plot(kind='bar', ax=ax, width=0.7)
ax.set_title('Event Volume by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Count')
ax.legend(title='Event Type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../figures/dow_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. User Behavior Analysis

In [ ]:
# Events per user
user_events = events.groupby('visitorid').size()

print("Events per user statistics:")
print(user_events.describe())

In [ ]:
# Distribution of events per user (log scale due to heavy tail)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(user_events.clip(upper=100), bins=50, edgecolor='white')
axes[0].set_title('Events per User (capped at 100)')
axes[0].set_xlabel('Number of Events')
axes[0].set_ylabel('Number of Users')

# Log-log plot
event_counts_dist = user_events.value_counts().sort_index()
axes[1].loglog(event_counts_dist.index, event_counts_dist.values, 'o', alpha=0.5)
axes[1].set_title('Events per User (Log-Log Scale)')
axes[1].set_xlabel('Number of Events (log)')
axes[1].set_ylabel('Number of Users (log)')

plt.tight_layout()
plt.savefig('../figures/user_activity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# User segmentation by activity level
def activity_segment(n_events):
    if n_events == 1:
        return 'Single Event'
    elif n_events <= 5:
        return 'Light (2-5)'
    elif n_events <= 20:
        return 'Medium (6-20)'
    elif n_events <= 100:
        return 'Heavy (21-100)'
    else:
        return 'Power (100+)'

user_segments = user_events.apply(activity_segment)
segment_order = ['Single Event', 'Light (2-5)', 'Medium (6-20)', 'Heavy (21-100)', 'Power (100+)']
segment_counts = user_segments.value_counts().reindex(segment_order)

print("User Activity Segments:")
for seg, count in segment_counts.items():
    pct = count / len(user_segments) * 100
    print(f"  {seg}: {count:,} ({pct:.1f}%)")

## 6. Conversion Funnel Analysis

In [ ]:
# Users at each stage of the funnel
viewers = events[events['event'] == 'view']['visitorid'].nunique()
carters = events[events['event'] == 'addtocart']['visitorid'].nunique()
purchasers = events[events['event'] == 'transaction']['visitorid'].nunique()

funnel_data = {
    'Stage': ['Viewed', 'Added to Cart', 'Purchased'],
    'Users': [viewers, carters, purchasers],
}
funnel_df = pd.DataFrame(funnel_data)
funnel_df['Conversion'] = funnel_df['Users'] / funnel_df['Users'].iloc[0] * 100
funnel_df['Stage Conversion'] = [100] + list((funnel_df['Users'].iloc[1:].values / funnel_df['Users'].iloc[:-1].values) * 100)

print("Conversion Funnel:")
print(funnel_df)

In [ ]:
# Visualize funnel
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.barh(funnel_df['Stage'], funnel_df['Users'], color=colors)

# Add value labels
for bar, users, conv in zip(bars, funnel_df['Users'], funnel_df['Conversion']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'{users:,} ({conv:.1f}%)', va='center')

ax.set_xlabel('Number of Users')
ax.set_title('Customer Conversion Funnel')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/conversion_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Transaction Analysis

In [ ]:
# Focus on transactions
transactions = events[events['event'] == 'transaction'].copy()
print(f"Total transactions: {len(transactions):,}")
print(f"Unique transaction IDs: {transactions['transactionid'].nunique():,}")
print(f"Unique purchasers: {transactions['visitorid'].nunique():,}")

In [ ]:
# Items per transaction
items_per_txn = transactions.groupby('transactionid')['itemid'].count()
print("Items per transaction:")
print(items_per_txn.describe())

In [ ]:
# Purchases per user
purchases_per_user = transactions.groupby('visitorid')['transactionid'].nunique()

print("\nPurchases per user:")
print(purchases_per_user.describe())

# One-time vs repeat buyers
one_time = (purchases_per_user == 1).sum()
repeat = (purchases_per_user > 1).sum()
print(f"\nOne-time buyers: {one_time:,} ({one_time/len(purchases_per_user)*100:.1f}%)")
print(f"Repeat buyers: {repeat:,} ({repeat/len(purchases_per_user)*100:.1f}%)")

## 8. Key Insights Summary

### Data Quality
- Dataset is clean with minimal missing values
- Transaction IDs available for tracking orders

### User Behavior
- Follows power law distribution (many light users, few heavy users)
- Clear conversion funnel from view → cart → purchase

### Time Patterns
- Distinct hourly and daily patterns
- Useful for timing retention campaigns

### Churn Signals
- High proportion of one-time buyers indicates churn risk
- Need to define churn window based on typical purchase frequency

In [ ]:
# Save key metrics for later use
key_metrics = {
    'total_events': len(events),
    'unique_visitors': events['visitorid'].nunique(),
    'unique_purchasers': purchasers,
    'conversion_rate': purchasers / viewers * 100,
    'repeat_buyer_rate': repeat / len(purchases_per_user) * 100,
    'avg_events_per_user': user_events.mean(),
    'date_range_days': (events['timestamp'].max() - events['timestamp'].min()).days,
}

print("Key Metrics:")
for k, v in key_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v:,}")